In [1]:
import torch, pprint


In [22]:
def inspect_torch_artifact(path):
    # 1) Try TorchScript first (scripted/traced models)
    try:
        m = torch.jit.load(path, map_location="cpu")
        print("✅ TorchScript module")
        print(m)  # prints graph + modules
        return
    except Exception:
        pass

    # 2) Otherwise treat it as a checkpoint/state_dict (pickle-based)
    #    Weights-only avoids executing pickled code (PyTorch ≥ 2.1).
    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=True)
    except Exception:
        # PyTorch < 2.1 (no weights_only arg)
        ckpt = torch.load(path, map_location="cpu", weights_only=False)

    print(f"✅ Loaded object type: {type(ckpt)}")

    if isinstance(ckpt, dict):
        print("\nTop-level keys:", list(ckpt.keys())[:20])
        # Common patterns: {"state_dict": ..., "epoch": ..., "optimizer": ...}
        sd = ckpt.get("state_dict", ckpt)
        if isinstance(sd, dict):
            keys = list(sd.keys())
            print(f"\nState dict param count: {len(keys)}")
            print("First 20 param keys:")
            pprint.pprint(keys[:20])
            # Show one tensor shape
            k = next((k for k,v in sd.items() if hasattr(v, "shape")), None)
            if k: print(f"\nExample tensor: {k} -> {sd[k].shape}")
    else:
        print("Value preview:", str(ckpt)[:500])

# usage:
# inspect_torch_artifact("ssl_pretrained_weights.pth")
# inspect_torch_artifact("swin_unetr.base_5000ep_f48_lr2e-4_pretrained.pt")

In [23]:
filepath = "/mnt/h/saranathan_share/ssl_pretrained_weights.pth"
filepath2 = "/mnt/h/saranathan_share/swin_unetr.base_5000ep_f48_lr2e-4_pretrained.pt"
inspect_torch_artifact(filepath2)

✅ Loaded object type: <class 'dict'>

Top-level keys: ['epoch', 'best_acc', 'state_dict']

State dict param count: 159
First 20 param keys:
['swinViT.patch_embed.proj.weight',
 'swinViT.patch_embed.proj.bias',
 'swinViT.layers1.0.blocks.0.norm1.weight',
 'swinViT.layers1.0.blocks.0.norm1.bias',
 'swinViT.layers1.0.blocks.0.attn.relative_position_bias_table',
 'swinViT.layers1.0.blocks.0.attn.relative_position_index',
 'swinViT.layers1.0.blocks.0.attn.qkv.weight',
 'swinViT.layers1.0.blocks.0.attn.qkv.bias',
 'swinViT.layers1.0.blocks.0.attn.proj.weight',
 'swinViT.layers1.0.blocks.0.attn.proj.bias',
 'swinViT.layers1.0.blocks.0.norm2.weight',
 'swinViT.layers1.0.blocks.0.norm2.bias',
 'swinViT.layers1.0.blocks.0.mlp.linear1.weight',
 'swinViT.layers1.0.blocks.0.mlp.linear1.bias',
 'swinViT.layers1.0.blocks.0.mlp.linear2.weight',
 'swinViT.layers1.0.blocks.0.mlp.linear2.bias',
 'swinViT.layers1.0.blocks.1.norm1.weight',
 'swinViT.layers1.0.blocks.1.norm1.bias',
 'swinViT.layers1.0.block

In [17]:
ckpt = torch.load(filepath2, map_location="cpu", weights_only=False)

In [19]:
check = ckpt['lr_scheduler']

KeyError: 'lr_scheduler'